In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from sklearn.utils.class_weight import compute_class_weight

print("🚀 Iniciando pipeline de predicción final (Producción)...")

# --- 1. FUNCIÓN DE LIMPIEZA (SOLO PARA LOS DATOS NUEVOS) ---
nltk.download('stopwords', quiet=True)
stemmer = PorterStemmer()
stop_words_nltk = set(stopwords.words('english'))

def preprocesamiento_stemming_basico(texto):
    if not isinstance(texto, str):
        return ""
    texto = str(texto).lower()
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    tokens = texto.split()
    tokens_stemmed = [stemmer.stem(p) for p in tokens if p not in stop_words_nltk]
    return " ".join(tokens_stemmed)

# --- 2. CARGAR DATOS ---
print("📥 Cargando datos preprocesados (100% del corpus de entrenamiento)...")
# Cargamos directamente el CSV con el trabajo ya hecho
df_entrenamiento = pd.read_csv("../Datos/Preprocesados/tcga_preprocesado.csv")

# Extraemos la columna ganadora y las etiquetas
X_train_texto = df_entrenamiento['text_stem_nltk'].astype(str)
mapeo_a_num = {'T1': 0, 'T2': 1, 'T3': 2, 'T4': 3}
y_train = df_entrenamiento['t'].map(mapeo_a_num)

print("📥 Cargando corpus nuevo para examinar...")
# OJO: Cambia esto por el nombre del archivo que te ha dado tu profesora
ruta_corpus_nuevo = "../Datos/Evaluacion/tcga_simple_dev_shuffled.csv" 
df_nuevo = pd.read_csv(ruta_corpus_nuevo)

# Limpiamos solo los textos de los pacientes nuevos
print("🧹 Limpiando los historiales nuevos...")
X_nuevo_texto = df_nuevo['text'].apply(preprocesamiento_stemming_basico)

# --- 3. TOKENIZACIÓN Y PADDING ---
print("🔢 Tokenizando...")
vocab_size = 5000
max_length = 300

# El tokenizer aprende SU diccionario de los datos de entrenamiento
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_texto) 

# Transformamos ambos textos a números usando ese diccionario
X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train_texto), maxlen=max_length, padding='post', truncating='post')
X_nuevo_pad = pad_sequences(tokenizer.texts_to_sequences(X_nuevo_texto), maxlen=max_length, padding='post', truncating='post')

# --- 4. ENTRENAMIENTO DEFINITIVO DE LA CNN ---
print("🧠 Entrenando la CNN definitiva con el 100% de la sabiduría disponible...")
pesos_clases = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
dict_pesos = dict(enumerate(pesos_clases))

modelo_cnn = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100, input_length=max_length),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

modelo_cnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Entrenamos (SIN validation_split) para que use todas las filas para aprender
modelo_cnn.fit(X_train_pad, y_train, epochs=10, batch_size=32, class_weight=dict_pesos, verbose=2)

# --- 5. PREDICCIÓN Y GENERACIÓN DEL CSV FINAL ---
print("🎯 Realizando diagnósticos sobre los pacientes nuevos...")
predicciones_prob = modelo_cnn.predict(X_nuevo_pad)
predicciones_num = np.argmax(predicciones_prob, axis=1)

# Diccionario inverso para traducir de vuelta a los estadios T
mapeo_a_texto = {0: 'T1', 1: 'T2', 2: 'T3', 3: 'T4'}

print("💾 Formateando y guardando el archivo final...")
# Machacamos la columna 't' de ejemplo con nuestra predicción real
df_nuevo['t'] = [mapeo_a_texto[num] for num in predicciones_num]

# Nos quedamos estrictamente con las 3 columnas que pide el formato
df_final = df_nuevo[['patient_id', 'text', 't']]

# Guardamos el CSV limpio y sin índice
nombre_archivo_final = "tcga_predicciones_entregar.csv"
df_final.to_csv(nombre_archivo_final, index=False)

print(f"✅ ¡Trabajo concluido! El archivo '{nombre_archivo_final}' está listo en tu carpeta.")

🚀 Iniciando pipeline de predicción final (Producción)...
📥 Cargando datos preprocesados (100% del corpus de entrenamiento)...
📥 Cargando corpus nuevo para examinar...
🧹 Limpiando los historiales nuevos...
🔢 Tokenizando...
🧠 Entrenando la CNN definitiva con el 100% de la sabiduría disponible...
Epoch 1/10


c:\Users\derad\Escritorio\IA\3\Segundo_cuatri\ProcesamientoLenguajeNatural\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


162/162 - 2s - 15ms/step - accuracy: 0.3463 - loss: 1.3125
Epoch 2/10
162/162 - 1s - 8ms/step - accuracy: 0.5120 - loss: 1.0571
Epoch 3/10
162/162 - 1s - 9ms/step - accuracy: 0.6132 - loss: 0.8498
Epoch 4/10
162/162 - 1s - 8ms/step - accuracy: 0.6933 - loss: 0.6893
Epoch 5/10
162/162 - 1s - 8ms/step - accuracy: 0.7730 - loss: 0.5427
Epoch 6/10
162/162 - 1s - 8ms/step - accuracy: 0.8371 - loss: 0.4077
Epoch 7/10
162/162 - 1s - 9ms/step - accuracy: 0.8901 - loss: 0.3043
Epoch 8/10
162/162 - 1s - 8ms/step - accuracy: 0.9203 - loss: 0.2270
Epoch 9/10
162/162 - 1s - 9ms/step - accuracy: 0.9453 - loss: 0.1712
Epoch 10/10
162/162 - 1s - 9ms/step - accuracy: 0.9610 - loss: 0.1274
🎯 Realizando diagnósticos sobre los pacientes nuevos...
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
💾 Formateando y guardando el archivo final...
✅ ¡Trabajo concluido! El archivo 'tcga_predicciones_entregar.csv' está listo en tu carpeta.
